In [1]:
import os
import cv2
import numpy as np
from tqdm import tqdm
import shutil

# ==========================================
# CONFIGURARE CĂI ȘI DIMENSIUNI
# ==========================================
# Luăm pozele curate, ne-augmentate, cu fundalul negru tăiat
INPUT_DIR = "../datasets/aptos_augmented_balanced"  # Aici am salvat pozele după ce le-am tăiat fundalul și le-am redimensionat

# Aici vom salva pozele transformate
OUTPUT_DIR = "../datasets/aptos_ben_graham"

# Dacă pozele tale sunt deja la 512, lasă 512.
IMG_SIZE = 1024

def delete_inside_folders():
    """Șterge folderele de Train, Val și Test din destinație pentru a începe cu un spațiu curat."""
    for split in ['train', 'val', 'test']:
        dst = os.path.join(OUTPUT_DIR, split)
        if os.path.exists(dst):
            print(f"🧹 Ștergem folderul '{split.upper()}' existent pentru a începe curat...")
            shutil.rmtree(dst)

def apply_ben_graham_filter(image, img_size=512):
    """
    Aplică formula originală a lui Ben Graham pentru Retinopatie Diabetică.
    """
    # 1. Ne asigurăm că imaginea are dimensiunea corectă
    img = cv2.resize(image, (img_size, img_size))
    
    # 2. Setăm parametrul de Blur (sigmaX). 
    # Regula de aur a lui Ben Graham: sigmaX = IMG_SIZE / 30
    sigmaX = img_size / 30.0 
    
    # 3. Aplicăm blur-ul gaussian
    blurred = cv2.GaussianBlur(img, (0, 0), sigmaX)
    
    # 4. Formula magică: image * 4 + blurred * (-4) + 128
    # Asta evidențiază extraordinar de bine vascularitățile!
    img_bg = cv2.addWeighted(img, 4, blurred, -4, 128)
    
    return img_bg

def process_dataset():
    print(f"🚀 Începem transformarea Ben Graham. Destinație: {OUTPUT_DIR}\n")
    
    foldere_split = ['train', 'val', 'test']
    clase = ['0', '1', '2', '3', '4']
    
    for split in foldere_split:
        for c in clase:
            src_folder = os.path.join(INPUT_DIR, split, c)
            dst_folder = os.path.join(OUTPUT_DIR, split, c)
            
            # Creăm structura de foldere
            os.makedirs(dst_folder, exist_ok=True)
            
            if not os.path.exists(src_folder):
                continue
                
            images = [f for f in os.listdir(src_folder) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
            if not images:
                continue
                
            # Procesăm imaginile
            for img_name in tqdm(images, desc=f"Procesare {split.upper()} / Clasa {c}", leave=False):
                src_img_path = os.path.join(src_folder, img_name)
                dst_img_path = os.path.join(dst_folder, img_name)
                
                # Citim imaginea
                img = cv2.imread(src_img_path)
                if img is None:
                    continue
                    
                # Aplicăm magia
                img_processed = apply_ben_graham_filter(img, img_size=IMG_SIZE)
                
                # Salvăm
                cv2.imwrite(dst_img_path, img_processed)

    print(f"\n✅ Gata! Datele filtrate se află în: {OUTPUT_DIR}")
    print("Deschide folderul și uită-te la câteva poze. Vei fi uimit de cum arată vasele de sânge!")

if __name__ == "__main__":
    delete_inside_folders()  # Ștergem folderele existente pentru a începe curat
    process_dataset()

🧹 Ștergem folderul 'TRAIN' existent pentru a începe curat...
🚀 Începem transformarea Ben Graham. Destinație: ../datasets/aptos_ben_graham




✅ Gata! Datele filtrate se află în: ../datasets/aptos_ben_graham
Deschide folderul și uită-te la câteva poze. Vei fi uimit de cum arată vasele de sânge!
